# Problem: Train an Autoencoder for Anomaly Detection

### Problem Statement
An **autoencoder** learns to reconstruct its input through a narrow **bottleneck**. If it only ever sees *normal* data, it learns to reconstruct normal inputs well and unfamiliar inputs poorly, so the **reconstruction error** of an input can be used as an anomaly score.

You will train a convolutional autoencoder on a single MNIST digit (the *normal* class, `0` by default) and treat every other digit as an anomaly. Labels are used **only** to select the normal training images and to evaluate the detector at the end. They are never used to train the model or to choose the threshold.

### Requirements
1. **Autoencoder**: `Autoencoder(latent_dim=32)`, with its two halves exposed as `self.encoder` and `self.decoder`.
   - **Encoder**: convolution → ReLU → max-pooling stages that shrink the image 28×28 → 14×14 → 7×7, then flatten and project to a `latent_dim`-dimensional code with `nn.Linear`. The code must be much smaller than the 784 input pixels. Without a bottleneck the network can learn something close to the identity map and reconstruct anomalies as well as normal digits.
   - **Decoder**: mirrors the encoder. An `nn.Linear` maps the code back to the 7×7 feature map, then transposed convolutions upsample 7×7 → 14×14 → 28×28.
   - The last layer is a **Sigmoid**. The inputs are pixel intensities in [0, 1] (`ToTensor()` with no `Normalize`), so the reconstruction has to be in the same range.
2. **Anomaly score**: `anomaly_scores(model, x)` returns the mean squared reconstruction error of **each image**, shape `(batch,)`. Average over channel, height and width, not over the batch. A single batch-mean loss cannot tell you *which* image is anomalous.
3. **Threshold**: hold out 10% of the normal training images as a validation set and set the threshold to the 95th percentile of their scores. By construction this flags about 5% of normal images (the false-positive rate you chose), and it does not require any anomalies.
4. **Evaluate** on the full MNIST test set:
   - the true-positive rate (the fraction of anomalies caught),
   - the false-positive rate (the fraction of normal digits flagged),
   - the threshold-free **AUROC**, which is the probability that a random anomaly scores higher than a random normal image.

### Constraints
- Train on normal images only. The validation split must not overlap the training split.
- Input shape is `(batch, 1, 28, 28)`. The reconstruction must have exactly the same shape.

### Things an interviewer may ask
- *Why must the bottleneck be narrow?* A wide enough autoencoder reconstructs anything, anomalies included. We tried a purely convolutional version of this network, whose 64×7×7 code holds 4× more numbers than the input, for 5 epochs with digit 0 as the normal class. It reached an AUROC of only about 0.73. Adding the 32-d linear bottleneck raised it to about 0.99.
- *Why pick the threshold on normal validation data?* You rarely have labelled anomalies, and tuning on test labels leaks the answer. A percentile of normal scores directly controls the false-positive rate.
- *Known failure mode*: reconstruction error is partly a measure of how much ink an image has. Simple digits (1, 7) reconstruct well even when they are anomalous, so some choices of normal digit work much worse than others. For example, with digit 8 as the normal class the AUROC drops to about 0.89.

<details>
  <summary>💡 Hint</summary>
  Track the spatial size: two 2×2 max-pools take 28 → 14 → 7, and <code>ConvTranspose2d(kernel_size=3, stride=2, padding=1, output_padding=1)</code> doubles it back (7 → 14 → 28). For the score, square the error and average it over every dimension except the batch dimension.
</details>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, random_split
import matplotlib.pyplot as plt

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        # Encoder: (B, 1, 28, 28) -> (B, latent_dim)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                        # 28x28 -> 14x14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                        # 14x14 -> 7x7
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, latent_dim),      # bottleneck: 784 pixels -> latent_dim numbers
        )
        # Decoder: (B, latent_dim) -> (B, 1, 28, 28), mirroring the encoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7),
            nn.ReLU(),
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # 7 -> 14
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1),   # 14 -> 28
            nn.Sigmoid(),                           # pixels are in [0, 1]
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
@torch.no_grad()
def anomaly_scores(model, x):
    """Per-image anomaly score: mean squared reconstruction error, shape (batch,)."""
    model.eval()
    recon = model(x)
    return ((recon - x) ** 2).flatten(1).mean(dim=1)  # average over C, H, W, never over the batch

In [ ]:
# Quick checks to run before training
ae = Autoencoder()
x = torch.rand(4, 1, 28, 28)
out = ae(x)
assert out.shape == x.shape, f"reconstruction {tuple(out.shape)} != input {tuple(x.shape)}"
assert out.min() >= 0 and out.max() <= 1, "end the decoder with a Sigmoid: pixels are in [0, 1]"
assert ae.encoder(x)[0].numel() < x[0].numel(), "the latent code must be smaller than the input"
scores = anomaly_scores(ae, x)
assert scores.shape == (4,), f"need one score per image, got shape {tuple(scores.shape)}"
assert torch.allclose(scores[1:2], anomaly_scores(ae, x[1:2])), \
    "an image's score must not depend on the other images in the batch"
print("All checks passed")

In [ ]:
# Pixels stay in [0, 1] (no Normalize) so they match the decoder's Sigmoid output
NORMAL_DIGIT = 0
transform = transforms.ToTensor()

train_full = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Train only on the normal class, and hold out 10% of it to calibrate the threshold
normal_idx = (train_full.targets == NORMAL_DIGIT).nonzero().squeeze(1).tolist()
normal_data = Subset(train_full, normal_idx)
n_val = len(normal_data) // 10
train_data, val_data = random_split(normal_data, [len(normal_data) - n_val, n_val],
                                    generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=256)
test_loader = DataLoader(test_dataset, batch_size=256)  # all ten digits
print(f"train: {len(train_data)} | val: {len(val_data)} (normal only) | test: {len(test_dataset)} (all digits)")

In [ ]:
torch.manual_seed(0)
model = Autoencoder(latent_dim=32)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
for epoch in range(epochs):
    model.train()
    running = 0.0
    for images, _ in train_loader:  # the labels are never used for training
        reconstructed = model(images)
        loss = criterion(reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item() * images.size(0)

    print(f"Epoch [{epoch + 1}/{epochs}], train MSE: {running / len(train_data):.4f}")

In [ ]:
def collect_scores(model, loader):
    scores, labels = [], []
    for images, y in loader:
        scores.append(anomaly_scores(model, images))
        labels.append(y)
    return torch.cat(scores), torch.cat(labels)


# Calibrate on held-out NORMAL images only. No anomalies and no test labels are used here.
val_scores, _ = collect_scores(model, val_loader)

threshold = torch.quantile(val_scores, 0.95).item()  # flags ~5% of normal images by construction
print(f"threshold = {threshold:.4f}")

In [ ]:
test_scores, test_labels = collect_scores(model, test_loader)
is_anomaly = test_labels != NORMAL_DIGIT  # ground truth, used only for evaluation
flagged = test_scores > threshold

tpr = flagged[is_anomaly].float().mean().item()   # anomalies caught (recall)
fpr = flagged[~is_anomaly].float().mean().item()  # normal digits wrongly flagged

# AUROC = P(random anomaly scores higher than random normal image); ties count half
diff = test_scores[is_anomaly][:, None] - test_scores[~is_anomaly][None, :]
auroc = ((diff > 0).float().mean() + 0.5 * (diff == 0).float().mean()).item()

print(f"TPR: {tpr:.3f} | FPR: {fpr:.3f} (target ~0.05) | AUROC: {auroc:.3f}")

In [ ]:
# The six most and six least anomalous test images (top row) and their reconstructions
order = test_scores.argsort(descending=True)
picks = torch.cat([order[:6], order[-6:]]).tolist()
images = torch.stack([test_dataset[i][0] for i in picks])
with torch.no_grad():
    recons = model(images)

fig, axes = plt.subplots(2, len(picks), figsize=(1.4 * len(picks), 3.2))
for j, i in enumerate(picks):
    axes[0, j].imshow(images[j, 0], cmap='gray')
    axes[0, j].set_title(f"{test_scores[i]:.3f}", fontsize=8)
    axes[1, j].imshow(recons[j, 0], cmap='gray')
    for ax in axes[:, j]:
        ax.axis('off')
fig.suptitle("Highest → lowest anomaly score (top: input, bottom: reconstruction)")
plt.show()